In [18]:
import numpy as np 
import pandas as pd 
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Linear Classifiers

#### Dataset

I am working with two 2010 census datasets from ICPSR. The dataset 'connectivity' includes features related to the road connectivity of each census tract. The dataset 'urbanicity' includes urban and rural population data, including a classification of each census tract as 'urban', 'rural', or 'mixed'. I made this into binary classification by setting urban against rural and mixed and pulled features from one of the datasets at a time.

In [3]:
# read data and filter out non-numeric values
connectivity = pd.read_csv("ICPSR_Connectivity2010.tsv", sep='\t', na_values=[' ']).dropna()
urbanicity = pd.read_csv("ICPSR_Urbanicity2010.tsv", sep='\t', na_values=[' ']).dropna()

connectivity_tracts = set(connectivity['TRACT_FIPS10'].to_list())
urbanicity_tracts = set(urbanicity['TRACT_FIPS10'].to_list())
tracts = connectivity_tracts & urbanicity_tracts # & polluting_tracts

connidx = connectivity['TRACT_FIPS10'].isin(tracts)
urbidx = urbanicity['TRACT_FIPS10'].isin(tracts)

connectivity = connectivity[connidx].drop_duplicates(subset='TRACT_FIPS10')
urbanicity = urbanicity[urbidx].drop_duplicates(subset='TRACT_FIPS10')

display(connectivity.columns)
display(urbanicity.columns)

Index(['TRACT_FIPS10', 'N_STREETS', 'SUM_STRINTLEN', 'N_NODES', 'N_REALNODES',
       'N_BLOCKS', 'TRACT_AREA_SQMILES', 'LINKNODERATIO', 'INTDENSITY',
       'STRNETDENSITY', 'CONNODERATIO', 'BLOCKDENSITY', 'AVGBLOCKLENGTH',
       'MEDBLOCKLENGTH', 'GAMMA', 'ALPHA'],
      dtype='str')

Index(['TRACT_FIPS10', 'TOT_POP_CENSUS', 'URBAN_POP', 'UA_POP', 'UC_POP',
       'RURAL_POP', 'TRACT_URBAN_PCT', 'TRACT_RURAL_PCT', 'URBANICITY',
       'TOT_POP_RUCA', 'LAND_AREA_SQMILES', 'POP_DENSITY', 'RUCA_PRIMARY_2010',
       'RUCA_SECONDARY_2010', 'RUCA_SECONDARYX10_2010', 'RUCA4', 'RUCA7'],
      dtype='str')

In [ ]:
# format as arrays
urbanicity_features = ['URBAN_POP', 'RURAL_POP', 'TRACT_URBAN_PCT', 'TRACT_RURAL_PCT', 'POP_DENSITY']
Xu = urbanicity[urbanicity_features].to_numpy()
Xc = connectivity.drop(columns=['TRACT_FIPS10']).to_numpy()
Y = urbanicity['URBANICITY'].to_numpy()
Y = np.where(Y==3, 0, 1) # 0: urban, 1: rural/mixed

#### Study

I decided to study two different things through this assignment:

1. The relationship between road connectivity and urbanicity, which I decided to study by observing the difference between model performance predicting urbanicity from urbanicity (directly related to urbanicity score) vs. connectivity (not directly related) datasets. 
2. The performance of Perceptron and SVM models on what I expected to be mostly linear (urbanicity) and less linear (connectivity) datasets.

I trained each model on each dataset and evaluated their accuracy on test sets. I ended up using SVM with a linear kernel, but I also tried poly and rbf kernels on the connectivity dataset.

In [ ]:
# split and normalize urbanicity dataset
X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(Xu, Y, test_size=0.33, random_state=42)
scaler_u = StandardScaler()
X_train_u = scaler_u.fit_transform(X_train_u)
X_test_u = scaler_u.transform(X_test_u)

In [ ]:
# perceptron performance with urbanicity
pct_urb = Perceptron(random_state=42, verbose=1)
pct_urb.fit(X_train_u, y_train_u)
pct_urb.score(X_test_u, y_test_u)

-- Epoch 1
Norm: 12.57, NNZs: 5, Bias: -5.000000, T: 48266, Avg. loss: 0.001411, Objective: 0.001411
Total training time: 0.01 seconds.
-- Epoch 2
Norm: 16.38, NNZs: 5, Bias: -6.000000, T: 96532, Avg. loss: 0.001127, Objective: 0.001127
Total training time: 0.02 seconds.
-- Epoch 3
Norm: 17.04, NNZs: 5, Bias: -6.000000, T: 144798, Avg. loss: 0.000203, Objective: 0.000203
Total training time: 0.02 seconds.
-- Epoch 4
Norm: 19.16, NNZs: 5, Bias: -6.000000, T: 193064, Avg. loss: 0.000682, Objective: 0.000682
Total training time: 0.03 seconds.
-- Epoch 5
Norm: 20.15, NNZs: 5, Bias: -7.000000, T: 241330, Avg. loss: 0.000542, Objective: 0.000542
Total training time: 0.04 seconds.
-- Epoch 6
Norm: 21.23, NNZs: 5, Bias: -7.000000, T: 289596, Avg. loss: 0.000481, Objective: 0.000481
Total training time: 0.05 seconds.
Convergence after 6 epochs took 0.05 seconds


0.9994952256761873

In [20]:
# SVM performance with urbanicity
svm_urb = LinearSVC(verbose=1)
svm_urb.fit(X_train_u, y_train_u)
svm_urb.score(X_test_u, y_test_u)

[LibLinear]iter  1 act 4.394e+04 pre 4.337e+04 delta 6.909e-01 f 4.827e+04 |g| 1.496e+05 CG   2
iter  2 act 1.436e+03 pre 1.123e+03 delta 6.909e-01 f 4.329e+03 |g| 1.375e+04 CG   3
iter  3 act 9.182e+02 pre 7.007e+02 delta 6.909e-01 f 2.893e+03 |g| 6.223e+03 CG   2
iter  4 act 5.887e+02 pre 4.441e+02 delta 6.909e-01 f 1.975e+03 |g| 2.858e+03 CG   2
iter  5 act 4.228e+02 pre 3.133e+02 delta 7.210e-01 f 1.386e+03 |g| 1.374e+03 CG   2
cg reaches trust region boundary
iter  6 act 3.048e+02 pre 2.335e+02 delta 1.107e+00 f 9.630e+02 |g| 6.505e+02 CG   2
cg reaches trust region boundary
iter  7 act 2.117e+02 pre 1.585e+02 delta 1.676e+00 f 6.582e+02 |g| 2.904e+02 CG   3
iter  8 act 1.384e+02 pre 1.056e+02 delta 2.397e+00 f 4.464e+02 |g| 1.285e+02 CG   2
iter  9 act 7.553e+01 pre 5.778e+01 delta 3.102e+00 f 3.080e+02 |g| 5.704e+01 CG   3
iter 10 act 3.144e+01 pre 2.592e+01 delta 3.102e+00 f 2.325e+02 |g| 2.431e+01 CG   3
iter 11 act 5.175e+00 pre 4.690e+00 delta 3.102e+00 f 2.010e+02 |g| 7.545

0.9997896773650781

In [ ]:
# split and normalize connectivity dataset
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(Xc, Y, test_size=0.33, random_state=42)
scaler_c = StandardScaler()
X_train_c = scaler_u.fit_transform(X_train_c)
X_test_c = scaler_u.transform(X_test_c)

In [ ]:
# perceptron performance with connectivity
pct_conn = Perceptron(random_state=42, verbose=1)
pct_conn.fit(X_train_c, y_train_c)
pct_conn.score(X_test_c, y_test_c)

-- Epoch 1
Norm: 49.25, NNZs: 15, Bias: -18.000000, T: 48266, Avg. loss: 0.836934, Objective: 0.836934
Total training time: 0.02 seconds.
-- Epoch 2
Norm: 64.69, NNZs: 15, Bias: -22.000000, T: 96532, Avg. loss: 0.817797, Objective: 0.817797
Total training time: 0.04 seconds.
-- Epoch 3
Norm: 50.08, NNZs: 15, Bias: -23.000000, T: 144798, Avg. loss: 0.729529, Objective: 0.729529
Total training time: 0.06 seconds.
-- Epoch 4
Norm: 73.91, NNZs: 15, Bias: -14.000000, T: 193064, Avg. loss: 0.700540, Objective: 0.700540
Total training time: 0.08 seconds.
-- Epoch 5
Norm: 90.49, NNZs: 15, Bias: -13.000000, T: 241330, Avg. loss: 0.892462, Objective: 0.892462
Total training time: 0.09 seconds.
-- Epoch 6
Norm: 94.59, NNZs: 15, Bias: -21.000000, T: 289596, Avg. loss: 0.864844, Objective: 0.864844
Total training time: 0.11 seconds.
-- Epoch 7
Norm: 39.24, NNZs: 15, Bias: -14.000000, T: 337862, Avg. loss: 0.761775, Objective: 0.761775
Total training time: 0.12 seconds.
-- Epoch 8
Norm: 80.79, NNZs:

0.9482606318091953

In [ ]:
# linear SVM performance with connectivity
svm_conn = LinearSVC(verbose=1, random_state=42)
svm_conn.fit(X_train_c, y_train_c)
svm_conn.score(X_test_c, y_test_c)

[LibLinear]iter  1 act 3.528e+04 pre 3.393e+04 delta 6.912e-01 f 4.827e+04 |g| 1.654e+05 CG   3
iter  2 act 2.773e+03 pre 2.904e+03 delta 6.912e-01 f 1.299e+04 |g| 3.259e+04 CG   4
iter  3 act 2.079e+03 pre 1.695e+03 delta 7.146e-01 f 1.021e+04 |g| 2.148e+04 CG   5
iter  4 act 7.030e+02 pre 5.657e+02 delta 9.142e-01 f 8.134e+03 |g| 5.387e+03 CG   7
iter  5 act 2.741e+02 pre 2.328e+02 delta 9.507e-01 f 7.431e+03 |g| 1.914e+03 CG   7
iter  6 act 4.406e+01 pre 4.124e+01 delta 9.507e-01 f 7.157e+03 |g| 7.474e+02 CG   9
cg reaches trust region boundary
iter  7 act 2.224e+01 pre 2.232e+01 delta 1.118e+00 f 7.113e+03 |g| 1.617e+02 CG  10
cg reaches trust region boundary
iter  8 act 2.051e+01 pre 2.109e+01 delta 1.128e+00 f 7.091e+03 |g| 1.403e+02 CG  12
cg reaches trust region boundary
iter  9 act 1.493e+01 pre 1.481e+01 delta 1.194e+00 f 7.070e+03 |g| 1.945e+02 CG  16
iter 10 act 1.782e+00 pre 1.745e+00 delta 1.194e+00 f 7.055e+03 |g| 1.198e+02 CG  11
iter 11 act 9.357e-01 pre 9.440e-01 delt

0.9566314726790898

In [31]:
# warning: takes a long time!
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_c, y_train_c)
display(svm_rbf.score(X_test_c, y_test_c))

svm_poly = SVC(kernel='poly', random_state=42)
svm_poly.fit(X_train_c, y_train_c)
display(svm_poly.score(X_test_c, y_test_c))

0.9578092794346528

0.9566314726790898

#### Results

| Dataset | Model | Test-Set Accuracy | Training Time |
|:---|:---|:---|:---|
| Urbanicity | Perceptron | 0.9995 | 6 epochs|
| Urbanicity | Linear SVM | 0.9998 | 11 iterations |
| Connectivity | Perceptron | 0.9483 | 12 epochs |
| Connectivity | Linear SVM | 0.9566 | 12 iterations |
| Connectivity | SVM (rbf) | 0.9578 | |
| Connectivity | SVM (poly) | 0.9566 | |

As I expected, both perceptron and SVM models performed very well on the urbanicity dataset, with the perceptron converging particularly quickly. Performance dropped a little on the connectivity dataset, but not as much as I expected- I think this means there is quite a direct relationship between road connectivity and urbanicity. SVM models performed only a little better than perceptron and tended to take longer, with rbf and poly kernels not really improving performance. Maybe this is because the data is already pretty linear.